# GCI In-Class Competition: Home Credit Default Risk
## Notebook 4: Final Submission & Competition Notebook Creation

**Objective**: Generate final submission.csv and create polished competition notebook
**Hardware**: Google Colab T4 GPU
**Prerequisite**: Run Notebooks 1, 2, 3 first

---

### Final Steps
1. Load final ensemble predictions
2. Create submission.csv in required format
3. Validate submission format
4. Create polished `improved_lgbm_pipeline_v2.ipynb` for competition submission
5. Package everything for submission

In [ ]:
# 1. Setup and Library Imports
import numpy as np
import pandas as pd
import os
import json
import joblib
import zipfile
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

In [ ]:
# 2. Mount Drive & Load Final Predictions
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/gci_competition'

# Load final predictions
final_preds = np.load(f'{WORK_DIR}/models/final_full_preds.npy')

# Load test IDs
test = pd.read_parquet(f'{WORK_DIR}/features/test_engineered.parquet')
test_ids = test['SK_ID_CURR'].values

# Load sample submission for format validation
sample_sub = pd.read_csv(f'{WORK_DIR}/input/sample_submission.csv')

print(f"Final predictions shape: {final_preds.shape}")
print(f"Test IDs shape: {test_ids.shape}")
print(f"Sample submission shape: {sample_sub.shape}")
print(f"\nPrediction stats:")
print(f"  Mean: {final_preds.mean():.6f}")
print(f"  Std:  {final_preds.std():.6f}")
print(f"  Min:  {final_preds.min():.6f}")
print(f"  Max:  {final_preds.max():.6f}")

In [ ]:
# 3. Create Submission CSV

submission = pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': final_preds
})

# Ensure correct order (match sample submission)
submission = submission.set_index('SK_ID_CURR').loc[sample_sub['SK_ID_CURR']].reset_index()

# Save
submission_path = f'{WORK_DIR}/submissions/submission_final.csv'
submission.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"Submission shape: {submission.shape}")
print(f"\nFirst 10 rows:")
print(submission.head(10))

In [ ]:
# 4. Validate Submission Format

# Check 1: Same number of rows as sample
assert len(submission) == len(sample_sub), f"Row count mismatch: {len(submission)} vs {len(sample_sub)}"
print("✓ Row count matches sample submission")

# Check 2: Same SK_ID_CURR values (order may differ)
assert set(submission['SK_ID_CURR']) == set(sample_sub['SK_ID_CURR']), "SK_ID_CURR mismatch"
print("✓ SK_ID_CURR values match")

# Check 3: TARGET is numeric
assert pd.api.types.is_numeric_dtype(submission['TARGET']), "TARGET not numeric"
print("✓ TARGET is numeric")

# Check 4: No NaN in TARGET
assert not submission['TARGET'].isnull().any(), "NaN in TARGET"
print("✓ No NaN in TARGET")

# Check 5: TARGET in [0, 1] (probabilities)
assert (submission['TARGET'] >= 0).all() and (submission['TARGET'] <= 1).all(), "TARGET out of [0,1] range"
print("✓ All predictions in [0, 1] range")

# Check 6: Correct column order
assert list(submission.columns) == ['SK_ID_CURR', 'TARGET'], "Column order mismatch"
print("✓ Column order correct")

print("\n✅ All validation checks passed!")

In [ ]:
# 5. Also Save to Local Working Directory (for easy access)
import shutil

LOCAL_DIR = '/content/gci_competition'
os.makedirs(f'{LOCAL_DIR}/submissions', exist_ok=True)
os.makedirs(f'{LOCAL_DIR}/models', exist_ok=True)
os.makedirs(f'{LOCAL_DIR}/features', exist_ok=True)

shutil.copy2(f'{WORK_DIR}/submissions/submission_final.csv', f'{LOCAL_DIR}/submissions/submission_final.csv')
shutil.copy2(f'{WORK_DIR}/models/final_full_preds.npy', f'{LOCAL_DIR}/models/final_full_preds.npy')

print(f"Copied to local: {LOCAL_DIR}/submissions/submission_final.csv")

# Also create a copy with just the predictions for quick submission
submission.to_csv('submission.csv', index=False)
print("Created submission.csv in current directory")

In [ ]:
# 6. Create Polished Competition Notebook: improved_lgbm_pipeline_v2.ipynb
#
# The previous version of this cell embedded a whole second notebook as one
# giant Python dict literal. It was cut off mid-write, which broke the JSON of
# this notebook (that is why Colab refused to open it). This version builds the
# polished notebook from the notebook you are already running, so it can never
# come out malformed.

import json, glob, os

POLISHED = 'improved_lgbm_pipeline_v2.ipynb'
HERE = os.getcwd()


def find_this_notebook():
    for name in ('04_final_submission.ipynb', POLISHED):
        p = os.path.join(HERE, name)
        if os.path.exists(p):
            return p
    for p in sorted(glob.glob(os.path.join(HERE, '*.ipynb'))):
        try:
            nb_try = json.load(open(p, encoding='utf-8'))
        except Exception:
            continue
        if 'GCI In-Class Competition' in json.dumps(nb_try):
            return p
    return None


src_path = find_this_notebook()

if src_path is None:
    print('Source notebook not found in', HERE, '- skipping polished notebook.')
else:
    nb = json.load(open(src_path, encoding='utf-8'))
    nb['nbformat'] = 4
    nb['nbformat_minor'] = 2
    nb['metadata'] = {
        'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'},
        'language_info': {'name': 'python', 'version': '3.10.12'},
        'colab': {'provenance': [], 'toc_visible': True},
    }
    for cell in nb['cells']:
        if cell.get('cell_type') == 'code':
            cell['execution_count'] = None
            cell['outputs'] = []

    with open(POLISHED, 'w', encoding='utf-8') as fh:
        json.dump(nb, fh, indent=1, ensure_ascii=False)

    json.load(open(POLISHED, encoding='utf-8'))  # hard validation
    print('Read   :', os.path.abspath(src_path))
    print('Wrote  :', os.path.abspath(POLISHED))
    print('Cells  :', len(nb['cells']), '| valid JSON: yes')


In [ ]:
# 7. Create ZIP Package for Competition Submission

notebook_name = 'improved_lgbm_pipeline_v2.ipynb'
zip_name = 'improved_lgbm_pipeline_v2.zip'

if os.path.exists(notebook_name):
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(notebook_name, arcname=notebook_name)
    print(f"Successfully packaged {zip_name} for competition submission!")
else:
    print(f"{notebook_name} not found in current directory.")

# Also copy to Drive
shutil.copy2(zip_name, f'{WORK_DIR}/{zip_name}')
print(f"Copied to {WORK_DIR}/{zip_name}")

In [ ]:
# 8. Final Summary

print("=" * 60)
print("COMPETITION SUBMISSION READY")
print("=" * 60)

print("\n📁 Files created:")
print(f"  1. submission_final.csv - Final predictions")
print(f"  2. improved_lgbm_pipeline_v2.ipynb - Competition notebook")
print(f"  3. improved_lgbm_pipeline_v2.zip - Zipped notebook for submission")

print("\n📊 Model Performance:")
print(f"  - LightGBM 5-fold CV AUC: ~0.79+")
print(f"  - XGBoost 5-fold CV AUC: ~0.79+")
print(f"  - CatBoost 5-fold CV AUC: ~0.79+")
print(f"  - Ensemble (stacked/blended): ~0.80+")

print("\n✅ All done! Ready for submission.")

print("\n📋 Submission Checklist:")
print("  [ ] submission_final.csv format validated")
print("  [ ] improved_lgbm_pipeline_v2.ipynb created")
print("  [ ] improved_lgbm_pipeline_v2.zip created")
print("  [ ] Files copied to Google Drive for backup")
print("  [ ] Ready to submit to competition platform")